# Validación del agente v3 · oficial, equipo y extra

Se parte de la v3 que Higinio añadió en `main` (`d065303`) y se incorpora un único
ajuste al prompt: aclarar las etiquetas de ingresos cuando la pregunta
no exige un identificador XBRL concreto. Se mantienen código de construcción,
modelos, embeddings, Qdrant y límites. No se añaden hooks ni reparaciones.

La pasada original completa obtuvo **18/20 en oficial, 19/20 en equipo y 12/12
en extra**. Su notebook ejecutado y el agente original se conservan en
`resultados/estudio_v002/v3_higinio_7a3f6e26fae6367e/`.

La ejecución siguiente vuelve a evaluar las 52 preguntas con el ajuste. Se
conservan las respuestas, las trazas y los fallos. Los conjuntos son conocidos
de desarrollo, no una prueba ciega de generalización.

## Resultado de la ejecución final

| Conjunto | V3 original | V3 con ajuste | Coste del agente (USD) | Latencia media (s) |
| --- | ---: | ---: | ---: | ---: |
| Oficial | 18/20 | **20/20** | 0.01988 | 14.33 |
| Equipo | 19/20 | **20/20** | 0.02482 | 18.21 |
| Extra | 12/12 | **12/12** | 0.00555 | 12.28 |

**Total: 52/52**, frente a 49/52 de la versión original. Los tres fallos
previos (`of-008`, `of-014`, `gjhh-012`) pasan en esta ejecución. Se conservan
**28/28 citas completas literales**, sin errores de ejecución. Coste total del
agente: **0.05025 USD**, sin embeddings de consulta.

Campaña final: `resultados/estudio_v002/v3_ajuste_73934c9b1f1be579/`.
Las seis celdas de código están ejecutadas y guardadas. Qdrant queda detenido
al terminar y los embeddings del corpus se mantienen sin cambios. Los resultados
corresponden a una pasada completa por configuración sobre conjuntos conocidos.


## Reglas de la versión evaluada

- **Selección de etiqueta:** si la pregunta no exige un identificador XBRL, se
  permite corregir una etiqueta elegida por el modelo usando los conceptos
  de ingresos: `Revenues` y `RevenueFromContractWithCustomerExcludingAssessedTax`.
  Un intento fallido no invalida el dato
  obtenido después con la etiqueta correcta.
- **Conceptos explícitos y cifras ausentes:** se respeta el identificador pedido;
  una ausencia confirmada no se cubre con texto, otra partida ni cálculos.
- **Abstención:** se devuelve `fuente="ninguna"`, con cifra, unidad, cita y
  fragmento nulos. Se aplica también cuando falta la empresa o el ejercicio.
- **Datos disponibles:** se conservan los importes completos y las unidades.
- **Comparaciones:** se consultan ambos ejercicios y se busca respaldo textual;
  la cifra principal corresponde al ejercicio más reciente.
- **Citas:** se comprueba también la transcripción completa del fragmento.

La celda siguiente muestra la regla exacta del agente importado. El agente recibe
solo la pregunta; las respuestas esperadas se reservan al evaluador.
No se añaden hooks, reparaciones ni cambios de prompt durante la evaluación.


In [1]:
import os, sys, json, hashlib
from pathlib import Path
inicio = Path.cwd().resolve()
RAIZ = next(p for p in (inicio, *inicio.parents) if (p / "src/taller_nlp").is_dir())
for p in (RAIZ, RAIZ / "src"):
    sys.path.insert(0, str(p))
os.environ.update(LANGSMITH_TRACING="false", LANGCHAIN_TRACING_V2="false")
import pandas as pd
from IPython.display import display
from experimentos.jchulvi import agente, agente_v2, agente_v3, qdrant_local
from taller_nlp import ManifiestoExperimento, CasoGolden
from taller_nlp.citas import cargar_fragmentos, normalizar_texto
from taller_nlp.hashing import calcular_sha256
assert os.getenv("OPENROUTER_API_KEY"), "Carga OPENROUTER_API_KEY fuera del notebook."
assert Path(agente_v3.__file__).resolve().is_relative_to(RAIZ)
print("Agente:", agente_v3.__file__)
print("Modelo:", agente.MODELO_CLOUD, "| embeddings:", agente.MODELO_EMBEDDINGS)


Agente: /Users/jchulvi/projects/Taller_NLP-jchulvi-qdrant-local/experimentos/jchulvi/agente_v3.py
Modelo: openrouter:deepseek/deepseek-v4-flash-0731 | embeddings: voyageai/voyage-4-lite


## 1. Infraestructura compartida y cambio de prompt

Se reutilizan los 1.749 vectores del corpus y los mismos modelos de v2. Las
consultas nuevas generan sus embeddings sin caché. Se conserva el límite de
seis llamadas al modelo y no se añaden hooks.

Una sesión prepara Qdrant y evalúa los tres conjuntos. Al terminar, se detiene
el contenedor de pruebas, cuya base no usa carpetas externas ni volúmenes.


In [2]:
indice = agente.preparar_embeddings_cloud()
qdrant = qdrant_local.QdrantLocal(
    agente.DIRECTORIO_RESULTADOS, ruta_indice=indice,
    corpus=agente.baseline.crear_corpus_baseline(),
    modelo=agente.MODELO_EMBEDDINGS, contrato=agente.CONTRATO_EMBEDDINGS,
)
constructor = agente_v3.crear_constructor()
assert constructor.middlewares == ()
assert constructor.configuracion.modelo == agente.MODELO_CLOUD
assert constructor.configuracion.max_iteraciones == 6
assert constructor.fabrica_herramientas.retriever.parametros["almacen"] == "qdrant"
assert constructor.configuracion.system_prompt == agente_v3.INSTRUCCIONES
assert agente_v3._REGLA_TEXTO_V2 not in agente_v3.INSTRUCCIONES
assert agente_v3._REGLA_TEXTO_V2 in agente_v2.INSTRUCCIONES
assert calcular_sha256(Path(agente_v3.__file__)) == "705771eba23d8e43800e12d630cfede84815c460c1fcc5950552866a48a6a638", "El agente difiere de la versión evaluada."
print("SHA256 agente_v3:", calcular_sha256(Path(agente_v3.__file__)))
print("Regla de abstención aplicada en v3:\n", agente_v3.REGLA_AUSENCIA_XBRL)
print("Middlewares propios:", len(constructor.middlewares))
print("Índice:", indice.name, "| dimensión:", qdrant.metadatos["dimension"])
print("SHA256 vectores:", qdrant.metadatos["sha256_vectores"])
print("Colección Qdrant:", qdrant.coleccion)



SHA256 agente_v3: 705771eba23d8e43800e12d630cfede84815c460c1fcc5950552866a48a6a638
Regla de abstención aplicada en v3:
 - Para toda pregunta que solicite una cifra, consulta primero el concepto
  exacto mediante get_xbrl_fact.
- Si la pregunta pide ingresos, ventas o facturación SIN indicar una etiqueta
  XBRL, las etiquetas a comprobar son Revenues y
  RevenueFromContractWithCustomerExcludingAssessedTax. Si una falta, consulta
  la otra antes de declarar ausencia. Usa el dato disponible del ejercicio
  pedido, aunque la primera consulta fallara; también en comparativas.
- Esa selección de etiqueta NO se permite si la pregunta exige un identificador
  XBRL literal: consulta ese concepto y aplica la abstención si falta, aunque
  exista otra etiqueta de ingresos. Las reglas siguientes siguen vigentes.
- Si get_xbrl_fact indica que la compañía no reportó el concepto solicitado en
  ese ejercicio, considera esa respuesta definitiva para la cifra pedida.
- Después de esa confirmación negati

## 2. Una pasada por conjunto

Los fallos se conservan. El identificador de campaña depende del código, los datos
y el índice: una modificación no reutiliza automáticamente respuestas de otra versión.
El agente recibe solamente la pregunta. El evaluador ve las respuestas esperadas.

El conjunto propio y el oficial tienen 20 preguntas cada uno. El extra contiene
12 preguntas numéricas sin dato autorizado; su tamaño se lee del JSONL.


In [3]:
DATASETS = {
    "extra": RAIZ / "golden_set_extra.jsonl",
    "oficial": RAIZ / "golden_set_oficial.jsonl",
    "equipo": RAIZ / "golden_set.jsonl",
}
archivos = [*sorted((RAIZ / "src/taller_nlp").glob("*.py")),
            RAIZ / "experimentos/baseline.py", Path(agente.__file__),
            Path(agente_v2.__file__), Path(agente_v3.__file__), Path(qdrant_local.__file__)]
identidad = {
    "codigo": {str(p.relative_to(RAIZ)): calcular_sha256(p) for p in archivos},
    "golden": {k: calcular_sha256(p) for k, p in DATASETS.items()},
    "vectores": qdrant.metadatos["sha256_vectores"],
    "corpus": [constructor.corpus.sha256_chunks, constructor.corpus.sha256_xbrl],
    "configuracion": constructor.configuracion.model_dump(mode="json"),
}
huella = hashlib.sha256(json.dumps(identidad, sort_keys=True).encode()).hexdigest()[:16]
CAMPANA = agente.DIRECTORIO_RESULTADOS / ("v3_ajuste_" + huella)
CAMPANA.mkdir(parents=True, exist_ok=True)
(CAMPANA / "configuracion.json").write_text(json.dumps(identidad, indent=2))
CASOS = {k: CasoGolden.cargar_jsonl(p, constructor.corpus, numero_esperado=20 if k != "extra" else None)
         for k, p in DATASETS.items()}
informes = {}
print("Campaña:", CAMPANA)
print("Preguntas por conjunto:", {k: len(casos) for k, casos in CASOS.items()})


def evaluar(conjunto):
    variante = constructor.con_ruta_progreso(CAMPANA / f"{conjunto}.progreso.json")
    informe = variante.construir().evaluar(DATASETS[conjunto])
    ManifiestoExperimento.desde_constructor(variante, informe=informe,
        metadatos=identidad).guardar(CAMPANA / f"{conjunto}.manifiesto.json")
    print(f"{conjunto}: {informe.aciertos_totales}/{informe.numero_preguntas}")
    return informe


Campaña: /Users/jchulvi/projects/Taller_NLP-jchulvi-qdrant-local/experimentos/jchulvi/resultados/estudio_v002/v3_ajuste_73934c9b1f1be579
Preguntas por conjunto: {'extra': 12, 'oficial': 20, 'equipo': 20}


In [4]:
with constructor.sesion_ejecucion():
    for conjunto in DATASETS:
        informes[conjunto] = evaluar(conjunto)

extra: 12/12


oficial: 20/20


equipo: 20/20


## 3. Resultados y regresión de los fallos previos

La nota usa el evaluador común (protocolo 5). Se muestra además la literalidad de
**toda** la cita: el evaluador solo verifica su prefijo normalizado de 120 caracteres.
Esta comprobación adicional no modifica ni repara ninguna respuesta.


In [5]:
chunks = cargar_fragmentos(constructor.corpus.ruta_chunks)
filas = []
for conjunto, informe in informes.items():
    for r in informe.resultados:
        a = r.respuesta_agente
        completa = (any(cid in chunks and normalizar_texto(a.cita) in normalizar_texto(chunks[cid].texto)
                        for cid in a.citas) if a.cita else None)
        filas.append({"conjunto": conjunto, "id": r.id_pregunta, "familia": r.familia, "acierto": r.acierto,
                      "fuente": a.fuente, "valor_cifra": a.cifra,
                      "cita_completa": completa, "cifra": r.cifra_correcta,
                      "trayectoria": r.trayectoria_correcta, "error": a.error,
                      "coste_usd": a.coste_usd, "segundos": a.latencia_ms / 1000,
                      "herramientas": len(a.llamadas)})
tabla = pd.DataFrame(filas)
resumen = tabla.groupby("conjunto").agg(evaluadas=("id", "size"), aciertos=("acierto", "sum"),
    errores=("error", "count"), coste_agente_usd=("coste_usd", lambda s: s.sum(min_count=len(s))),
    latencia_mediana_s=("segundos", "median"), latencia_p95_s=("segundos", lambda s: s.quantile(.95)))
resumen["esperadas"] = pd.Series({k: len(casos) for k, casos in CASOS.items()})
assert (resumen.evaluadas == resumen.esperadas).all(), "Faltan preguntas por evaluar."
resumen["tasa_acierto"] = resumen.aciertos / resumen.evaluadas
display(resumen)
display(tabla[tabla.id.isin(["of-001", "of-004", "of-005", "of-008", "of-014", "gjhh-012"])])
display(tabla)
tabla.to_csv(CAMPANA / "resultados.csv", index=False)
print("Aciertos totales:", int(tabla.acierto.sum()), "/", len(tabla))
print("Todos los conjuntos sin fallos:", bool((resumen.aciertos == resumen.esperadas).all()))
print("Fallos del conjunto extra:")
display(tabla[(tabla.conjunto == "extra") & ~tabla.acierto])
print("Qdrant detenido:", not qdrant.en_marcha)


,evaluadas,aciertos,errores,coste_agente_usd,latencia_mediana_s,latencia_p95_s,esperadas,tasa_acierto
conjunto,,,,,,,,
equipo,20,20,0,0.024821,12.366237,51.857225,20,1.0
extra,12,12,0,0.005549,11.452820,20.453872,12,1.0
oficial,20,20,0,0.019881,14.457465,27.636597,20,1.0


,conjunto,id,familia,acierto,fuente,valor_cifra,cita_completa,cifra,trayectoria,error,coste_usd,segundos,herramientas
12,oficial,of-001,extractiva,True,texto,NaN,True,None,True,None,0.001456,21.762312,2
15,oficial,of-004,extractiva,True,texto,NaN,True,None,True,None,0.001054,19.387971,1
16,oficial,of-005,extractiva,True,texto,NaN,True,None,True,None,0.001250,14.670726,2
19,oficial,of-008,numerica,True,xbrl,4.161610e+11,None,True,True,None,0.000493,9.102032,3
25,oficial,of-014,comparativa,True,ambas,2.817240e+11,True,True,True,None,0.001248,15.598278,6
43,equipo,gjhh-012,numerica,True,ambas,7.169240e+11,True,True,True,None,0.001527,51.655615,4


,conjunto,id,familia,acierto,fuente,valor_cifra,cita_completa,cifra,trayectoria,error,coste_usd,segundos,herramientas
0,extra,extra-001,numerica,True,ninguna,NaN,None,True,True,None,0.000539,18.944552,2
1,extra,extra-002,numerica,True,ninguna,NaN,None,True,True,None,0.000491,6.256459,2
2,extra,extra-003,numerica,True,ninguna,NaN,None,True,True,None,0.000440,13.821246,2
3,extra,extra-004,numerica,True,ninguna,NaN,None,True,True,None,0.000413,6.945457,2
4,extra,extra-005,numerica,True,ninguna,NaN,None,True,True,None,0.000427,13.310980,1
5,extra,extra-006,numerica,True,ninguna,NaN,None,True,True,None,0.000465,22.298595,1
6,extra,extra-007,numerica,True,ninguna,NaN,None,True,True,None,0.000470,7.028411,2
7,extra,extra-008,numerica,True,ninguna,NaN,None,True,True,None,0.000572,9.378370,2
8,extra,extra-009,numerica,True,ninguna,NaN,None,True,True,None,0.000458,17.918264,2
9,extra,extra-010,numerica,True,ninguna,NaN,None,True,True,None,0.000395,5.986793,2


Aciertos totales: 52 / 52
Todos los conjuntos sin fallos: True
Fallos del conjunto extra:


,conjunto,id,familia,acierto,fuente,valor_cifra,cita_completa,cifra,trayectoria,error,coste_usd,segundos,herramientas


Qdrant detenido: True


## Diagnóstico y ajuste del prompt

La v3 original falló `of-008`, `of-014` y `gjhh-012`: confundió la ausencia de
`Revenues`, etiqueta elegida por el modelo, con la ausencia de ingresos.
La herramienta ofrecía `RevenueFromContractWithCustomerExcludingAssessedTax`.
En el caso de Amazon, incluso obtuvo el dato con esa etiqueta, pero mantuvo la
abstención por el primer intento fallido.

El cambio añade ocho líneas al prompt de Higinio: comprobar ambas etiquetas de
ingresos cuando la pregunta no exige un identificador concreto, aprovechar el
dato obtenido y conservar la abstención estricta para conceptos explícitos.
No se incorporan empresas, importes ni respuestas de los golden sets al prompt.

Un intento intermedio solo indicaba que se podía corregir la etiqueta y volvió
a fallar las dos preguntas oficiales. Se interrumpió; sus checkpoints se
conservan en `v3_ajuste_5d6a5c5c162ce961`. La pasada final comprueba los tres
conjuntos desde cero con la aclaración de las etiquetas.

El extra mantiene sus criterios: nueve conceptos ausentes y tres ejercicios
fuera del corpus. Se exige `fuente="ninguna"`, cifra y unidad nulas y las
herramientas esperadas. No se repiten selectivamente los fallos ni se alteran
los veredictos.


## 4. Respuestas y evidencia para revisión

Estos son conjuntos conocidos de desarrollo, no una evaluación ciega. Un 20/20
mide los criterios del evaluador; no garantiza corrección semántica de cada frase.
Se conserva el detalle para revisar esa diferencia. El coste del agente excluye
los embeddings de consulta.


In [6]:
for conjunto, informe in informes.items():
    preguntas = {c.id: c for c in CASOS[conjunto]}
    for r in informe.resultados:
        a = r.respuesta_agente
        print(f"\n{'=' * 70}\n{conjunto} · {r.id_pregunta} · {'ACIERTO' if r.acierto else 'FALLO'}")
        print("PREGUNTA:", preguntas[r.id_pregunta].pregunta)
        print("REFERENCIA:", preguntas[r.id_pregunta].respuesta_esperada)
        print("RESPUESTA:", a.respuesta)
        print("CIFRA/UNIDAD:", a.cifra, a.unidad, "| FUENTE:", a.fuente)
        print("CITA:", a.cita, "| CHUNKS:", a.citas)
        for llamada in a.llamadas:
            print("HERRAMIENTA:", llamada.nombre, json.dumps(llamada.argumentos, ensure_ascii=False))
        print("OBSERVACIONES:", r.observaciones, "| ERROR:", a.error)



extra · extra-001 · ACIERTO
PREGUNTA: ¿Cuál fue el beneficio bruto de Amazon en el ejercicio fiscal 2025?
REFERENCIA: El beneficio bruto de Amazon no está disponible como hecho XBRL autorizado en el corpus para FY2025; fuente="ninguna".
RESPUESTA: El beneficio bruto (GrossProfit) de Amazon no está reportado como hecho XBRL en el corpus para el ejercicio fiscal 2025.
CIFRA/UNIDAD: None None | FUENTE: ninguna
CITA: None | CHUNKS: ()
HERRAMIENTA: list_available {}
HERRAMIENTA: get_xbrl_fact {"ticker": "AMZN", "fiscal_year": 2025, "concept": "GrossProfit"}
OBSERVACIONES: () | ERROR: None

extra · extra-002 · ACIERTO
PREGUNTA: ¿Cuál era el total de pasivos de Amazon al cierre del ejercicio fiscal 2024?
REFERENCIA: El total de pasivos de Amazon no está disponible como hecho XBRL autorizado en el corpus para FY2024; fuente="ninguna".
RESPUESTA: Amazon no reportó el concepto de pasivos (Liabilities) en el ejercicio fiscal 2024 en el corpus XBRL disponible, por lo que no es posible indicar el 